In [2]:
import json

# Automatically generate relationships.json in the current working directory
relationships_data = {
    "father": [
        {"John": "William"},
        {"John": "David"},
        {"John": "Adam"},
        {"William": "Chris"},
        {"William": "Stephanie"},
        {"David": "Wayne"},
        {"David": "Tiffany"},
        {"David": "Julie"},
        {"David": "Neil"},
        {"David": "Peter"},
        {"Adam": "Sophia"},
    ],
    "mother": [
        {"Megan": "William"},
        {"Megan": "David"},
        {"Megan": "Adam"},
        {"Emma": "Stephanie"},
        {"Emma": "Chris"},
        {"Olivia": "Tiffany"},
        {"Olivia": "Julie"},
        {"Olivia": "Neil"},
        {"Olivia": "Peter"},
        {"Lily": "Sophia"},
    ],
}

with open("relationships.json", "w") as f:
    json.dump(relationships_data, f, indent=4)

print(
    "✅ relationships.json has been created successfully in the current folder!"
)

✅ relationships.json has been created successfully in the current folder!


In [3]:
import json
from kanren import Relation, facts, run, var, conde
from kanren.core import lall

# 1. Define Predicates / Relations
father = Relation()
mother = Relation()

# 2. Ingest JSON Knowledge Base
with open("relationships.json") as f:
    d = json.load(f)

for item in d["father"]:
    facts(father, (list(item.keys())[0], list(item.values())[0]))

for item in d["mother"]:
    facts(mother, (list(item.keys())[0], list(item.values())[0]))

# 3. Define Relational Rules
def parent(p, c):
    """p is parent of c if p is father or mother of c"""
    return conde([father(p, c)], [mother(p, c)])

def grandparent(gp, c):
    """gp is grandparent of c if gp is parent of intermediate p, and p is parent of c"""
    p = var()
    return lall(parent(gp, p), parent(p, c))

def sibling(a, b):
    """a and b are siblings if they share the same parent p"""
    p = var()
    return lall(parent(p, a), parent(p, b))

def uncle(u, nibling):
    """u is uncle of nibling if u is male and a sibling of nibling's parent"""
    p = var()
    return lall(father(u, var()), sibling(u, p), parent(p, nibling))

def spouse(h, w):
    """h and w are spouses if they share a common child c"""
    c = var()
    return lall(father(h, c), mother(w, c))

# --- Interrogation Queries (Lab Sheet Questions 1, 2, and 3) ---
q = var()

# Question 1: Find David's siblings
david_siblings = set(run(0, q, sibling(q, "David"))) - {"David"}
print(f"David's Siblings: {david_siblings}")

# Question 2: Find Tiffany's uncles
tiffany_uncles = set(run(0, q, uncle(q, "Tiffany")))
print(f"Tiffany's Uncles: {tiffany_uncles}")

# Question 3: Find all spouses
spouses = run(0, (var("h"), var("w")), spouse(var("h"), var("w")))
print(f"All Spouses: {set(spouses)}")

David's Siblings: {'William', 'Adam'}
Tiffany's Uncles: {'William', 'Adam', 'David'}
All Spouses: {('David', 'Olivia'), ('William', 'Emma'), ('Adam', 'Lily'), ('John', 'Megan')}


In [4]:
from kanren import Relation, facts, run, var
from kanren.core import lall

# 1. Define Logic Relations
adjacent = Relation()
coastal = Relation()

# 2. Ingest Adjacent States
with open("adjacent_states.txt") as f:
    for line in f:
        parts = [item.strip() for item in line.strip().split(",") if item.strip()]
        if len(parts) > 1:
            state = parts[0]
            for neighbor in parts[1:]:
                facts(adjacent, (state, neighbor))

# 3. Ingest Coastal States
with open("coastal_states.txt") as f:
    for line in f:
        for c_state in line.strip().split(","):
            c_clean = c_state.strip()
            if c_clean:
                facts(coastal, c_clean)

# --- Queries from Lab Sheet ---
x = var()

# Example 1: Is Nevada adjacent to Louisiana?
res1 = run(0, x, adjacent("Nevada", "Louisiana"))
print(f"Example 1 - Is Nevada adjacent to Louisiana? {'Yes' if res1 else 'No'}")

# Example 2: States adjacent to Oregon
res2 = run(0, x, adjacent("Oregon", x))
print(f"Example 2 - States adjacent to Oregon: {list(res2)}")

# Example 3: States adjacent to Mississippi that are coastal
res3 = run(0, x, lall(adjacent("Mississippi", x), coastal(x)))
print(f"Example 3 - Coastal states adjacent to Mississippi: {list(res3)}")

# Question 1 (Extension): States adjacent to BOTH Colorado and Texas
res4 = run(0, x, lall(adjacent("Colorado", x), adjacent("Texas", x)))
print(f"Extension Query - States adjacent to both Colorado and Texas: {list(res4)}")

Example 1 - Is Nevada adjacent to Louisiana? No
Example 2 - States adjacent to Oregon: ['Nevada', 'Idaho', 'Washington', 'California']
Example 3 - Coastal states adjacent to Mississippi: []
Extension Query - States adjacent to both Colorado and Texas: ['New Mexico', 'Oklahoma']


In [5]:
def select_clothing(location, season, weather="Clear"):
    # Rule 1: Sub-Zero / Tundra
    if location in ["Tundra", "Glacial_Peak", "Arctic"] or season == "Winter":
        return {
            "Outerwear": "Heavy Thermal Parka",
            "Headwear": "Insulated Fur Balaclava",
            "Footwear": "Crampon Boots",
        }
    # Rule 2: Arid / Desert
    elif location in ["Sahara_Dunes", "Wasteland"] and season == "Summer":
        return {
            "Outerwear": "Breathable Linen Robes",
            "Headwear": "UV Shemagh Scarf",
            "Footwear": "Leather Sandals",
        }
    # Rule 3: Rainforest / Monsoon
    elif location == "Rainforest" or weather == "Torrential_Downpour":
        return {
            "Outerwear": "Waterproof Waxed Poncho",
            "Headwear": "Wide-Brimmed Hat",
            "Footwear": "Treated Muck Boots",
        }
    # Rule 4: Temperate / Urban
    else:
        return {
            "Outerwear": "Reinforced Leather Jacket",
            "Headwear": "Standard Hood",
            "Footwear": "Combat Boots",
        }


# Test evaluation
char1 = select_clothing(
    location="Tundra", season="Winter", weather="Snowstorm"
)
char2 = select_clothing(location="Sahara_Dunes", season="Summer")

print(f"Character 1 Outfit (Tundra/Winter): {char1}")
print(f"Character 2 Outfit (Desert/Summer): {char2}")

Character 1 Outfit (Tundra/Winter): {'Outerwear': 'Heavy Thermal Parka', 'Headwear': 'Insulated Fur Balaclava', 'Footwear': 'Crampon Boots'}
Character 2 Outfit (Desert/Summer): {'Outerwear': 'Breathable Linen Robes', 'Headwear': 'UV Shemagh Scarf', 'Footwear': 'Leather Sandals'}


In [6]:
# Semantic Network Knowledge Representation via Dictionary Triples
semantic_net = {
    "Mammal": {
        "is_a": "Living Fauna",
        "has_property": ["Fur/Hair", "Mammary Glands", "Dense Bone Structure"],
        "reproduction": "Viviparous (Live Birth)",
        "locomotion": "Quadrupedal / Terrestrial",
    },
    "Bird": {
        "is_a": "Living Fauna",
        "has_property": [
            "Feathers/Plumage",
            "Toothless Beak",
            "Hollow Skeleton",
        ],
        "reproduction": "Oviparous (Egg Laying)",
        "locomotion": "Bipedal / Winged Flight",
    },
}


def classify_wildlife_feature(detected_features: list) -> str:
    """Classifies a detection candidate based on semantic feature matching."""
    scores = {"Mammal": 0, "Bird": 0}

    for category, attributes in semantic_net.items():
        for feat in detected_features:
            if feat in attributes["has_property"]:
                scores[category] += 1

    return max(scores, key=scores.get)


# Test Scenarios
sample_1 = ["Feathers/Plumage", "Toothless Beak"]
sample_2 = ["Fur/Hair", "Dense Bone Structure"]

print(
    f"Vision Feature Vector {sample_1} --> Classified as: {classify_wildlife_feature(sample_1)}"
)
print(
    f"Vision Feature Vector {sample_2} --> Classified as: {classify_wildlife_feature(sample_2)}"
)

Vision Feature Vector ['Feathers/Plumage', 'Toothless Beak'] --> Classified as: Bird
Vision Feature Vector ['Fur/Hair', 'Dense Bone Structure'] --> Classified as: Mammal
